In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [ ]:
from google.colab import files
uploaded=files.upload

In [ ]:
spark = SparkSession.builder \
    .appName("SmartHomeEnergyETL") \
    .getOrCreate()

In [ ]:
energy_df = spark.read.csv(
    "energy_usage.csv",
    header=True,
    inferSchema=True
)

In [ ]:
energy_df = energy_df.withColumn(
    "timestamp",
    to_timestamp("timestamp")
)

In [ ]:
energy_df = energy_df.withColumn(
    "date",
    to_date("timestamp")
)

In [ ]:
energy_df = energy_df.withColumn(
    "week",
    weekofyear("timestamp")
)

In [ ]:
daily_summary = (
    energy_df.groupBy("date")
    .agg(
        sum("energy_kwh").alias("total_energy"),
        avg("energy_kwh").alias("average_energy"),
        max("energy_kwh").alias("peak_energy")
    )
)
daily_summary.show()

+----------+------------+------------------+-----------+
|      date|total_energy|    average_energy|peak_energy|
+----------+------------+------------------+-----------+
|2026-06-03|        18.2|3.6399999999999997|        5.3|
|2026-06-02|        17.5|               3.5|        4.7|
|2026-06-01|        16.6|3.3200000000000003|        5.0|
|2026-06-04|        18.5|               3.7|        5.1|
+----------+------------+------------------+-----------+



In [ ]:
weekly_summary = (
    energy_df.groupBy("week")
    .agg(
        sum("energy_kwh").alias("weekly_energy"),
        avg("energy_kwh").alias("average_energy")
    )
)
weekly_summary.show()

+----+-------------+--------------+
|week|weekly_energy|average_energy|
+----+-------------+--------------+
|  23|         70.8|          3.54|
+----+-------------+--------------+



In [ ]:
daily_summary.toPandas().to_csv(
    "daily_energy_summary.csv",
    index=False
)

weekly_summary.toPandas().to_csv(
    "weekly_energy_summary.csv",
    index=False
)